<a href="https://colab.research.google.com/github/sharifovlab/Params-fine-tuning-comparison/blob/main/Params_fine_tuning_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 9.3 MB/s eta 0:00:00


In [ ]:
import time
import warnings

import optuna
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

In [ ]:
RANDOM_STATE = 42
N_ITER = 36
CV_FOLDS = 5

In [ ]:
X.shape, y.shape

((569, 30), (569,))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
SCORING = "roc_auc"

In [ ]:
def evaluate(name, model, search_time):
    start = time.perf_counter()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predict_time = time.perf_counter() - start

    return {
        "method": name,
        "search_time_sec": round(search_time, 2),
        "predict_time_sec": round(predict_time, 4),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "best_params": model.get_params(),
    }


results = []

In [ ]:
# 2. Basic XGBoost
print("Training Basic XGBoost...")
start = time.perf_counter()

basic_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
basic_xgb.fit(X_train, y_train)

basic_time = time.perf_counter() - start
results.append(evaluate("Basic XGBoost", basic_xgb, basic_time))

Training Basic XGBoost...


In [ ]:
# 3. GridSearchCV
print("Running GridSearchCV...")
grid_params = {
    "n_estimators": [100, 200],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_grid=grid_params,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start

results.append(evaluate("GridSearchCV", grid_search.best_estimator_, grid_time))
print("GridSearch best params:", grid_search.best_params_)
print("GridSearch best CV ROC-AUC:", grid_search.best_score_)


Running GridSearchCV...
GridSearch best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}
GridSearch best CV ROC-AUC: 0.9943240454076367


In [ ]:
# 4. RandomizedSearchCV
print("Running RandomizedSearchCV...")
random_params = {
    "n_estimators": [50, 100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6, 8],
    "learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 2, 5, 10],
    "reg_lambda": [0.1, 1, 5, 10],
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=random_params,
    n_iter=N_ITER,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

start = time.perf_counter()
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start

results.append(evaluate("RandomizedSearchCV", random_search.best_estimator_, random_time))
print("RandomizedSearch best params:", random_search.best_params_)
print("RandomizedSearch best CV ROC-AUC:", random_search.best_score_)

Running RandomizedSearchCV...
RandomizedSearch best params: {'subsample': 0.7, 'reg_lambda': 0.1, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
RandomizedSearch best CV ROC-AUC: 0.9946336429308564


In [ ]:
# 5. Optuna
print("Running Optuna...")


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
    }

    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, proba))

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

start = time.perf_counter()
study.optimize(objective, n_trials=N_ITER)
optuna_time = time.perf_counter() - start

best_optuna_xgb = XGBClassifier(
    **study.best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_optuna_xgb.fit(X_train, y_train)

results.append(evaluate("Optuna", best_optuna_xgb, optuna_time))
print("Optuna best params:", study.best_params)
print("Optuna best CV ROC-AUC:", study.best_value)


Running Optuna...
Optuna best params: {'n_estimators': 197, 'max_depth': 8, 'learning_rate': 0.1600604091630167, 'subsample': 0.880320701617949, 'colsample_bytree': 0.6889426382810098, 'min_child_weight': 1, 'reg_lambda': 0.48659783676640456}
Optuna best CV ROC-AUC: 0.9941176470588236


In [ ]:
# 6. Final comparison
results_df = pd.DataFrame(results)
metric_columns = [
    "method",
    "search_time_sec",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
]

print("\n================ FINAL TEST-SET COMPARISON ================")
print(results_df[metric_columns].sort_values("test_roc_auc", ascending=False).to_string(index=False))

results_df.to_csv("xgb_search_benchmark_results.csv", index=False)
print("\nSaved: xgb_search_benchmark_results.csv")


================ FINAL TEST-SET COMPARISON ================
            method  search_time_sec  test_accuracy  test_precision  test_recall  test_f1  test_roc_auc
      GridSearchCV            52.45        0.95614        0.946667     0.986111 0.965986      0.994378
     Basic XGBoost             0.14        0.95614        0.946667     0.986111 0.965986      0.992725
            Optuna            36.41        0.95614        0.946667     0.986111 0.965986      0.992725
RandomizedSearchCV            35.27        0.95614        0.946667     0.986111 0.965986      0.992063

Saved: xgb_search_benchmark_results.csv


Это не означает, что Optuna плохой или что тюнинг бесполезен. Это означает, что Breast Cancer — неподходящий датасет, чтобы увидеть разницу между методами оптимизации. Он слишком мал, слишком чист и слишком легко решается.

Чтобы увидеть реальную разницу, нужен более сложный датасет:
* больше строк, например 10k–100k;

* больше признаков;

* нелинейные взаимодействия;

* категориальные переменные;

* class imbalance;

* более шумная целевая переменная.




**тот же pipeline на ML-ready Telco Customer Churn. Там baseline XGBoost не будет настолько близок к потолку,**

**Restart the session**

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [ ]:
!kaggle datasets download -d blastchar/telco-customer-churn -p /content/data --unzip
!ls -lh /content/data

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100% 172k/172k [00:00<00:00, 328kB/s]

total 956K
-rw-r--r-- 1 root root 955K Oct 21 06:07 WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
RANDOM_STATE = 42
N_ITER = 36
CV_FOLDS = 5
SCORING = "roc_auc"

In [ ]:
import time
import warnings

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# 1. Load and prepare data
# -----------------------------
import pandas as pd
df = pd.read_csv("/content/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"]

print("Dataset shape:", df.shape)
print("Churn distribution:")
print(y.value_counts(normalize=True))


Dataset shape: (7043, 20)
Churn distribution:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

results = []


In [ ]:
def evaluate(name, model, search_time):
    start = time.perf_counter()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predict_time = time.perf_counter() - start

    cm = confusion_matrix(y_test, y_pred)

    return {
        "method": name,
        "search_time_sec": round(search_time, 2),
        "predict_time_sec": round(predict_time, 4),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "true_negatives": cm[0, 0],
        "false_positives": cm[0, 1],
        "false_negatives": cm[1, 0],
        "true_positives": cm[1, 1],
        "best_params": model.get_params(),
    }


In [ ]:
# -----------------------------
# 2. Basic XGBoost
# -----------------------------
print("\nTraining Basic XGBoost...")
start = time.perf_counter()

basic_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
basic_xgb.fit(X_train, y_train)

basic_time = time.perf_counter() - start
results.append(evaluate("Basic XGBoost", basic_xgb, basic_time))


Training Basic XGBoost...


In [ ]:
# -----------------------------
# 3. GridSearchCV
# -----------------------------
print("Running GridSearchCV...")
grid_params = {
    "n_estimators": [100, 200],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_grid=grid_params,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - start

results.append(evaluate("GridSearchCV", grid_search.best_estimator_, grid_time))

print("GridSearch best params:", grid_search.best_params_)
print("GridSearch best CV ROC-AUC:", grid_search.best_score_)


Running GridSearchCV...
GridSearch best params: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100, 'subsample': 1.0}
GridSearch best CV ROC-AUC: 0.8499851798881133


In [ ]:
# -----------------------------
# 4. RandomizedSearchCV
# -----------------------------
print("Running RandomizedSearchCV...")
random_params = {
    "n_estimators": [50, 100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6, 8],
    "learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 2, 5, 10],
    "reg_lambda": [0.1, 1, 5, 10],
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=random_params,
    n_iter=N_ITER,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

start = time.perf_counter()
random_search.fit(X_train, y_train)
random_time = time.perf_counter() - start

results.append(
    evaluate("RandomizedSearchCV", random_search.best_estimator_, random_time)
)

print("RandomizedSearch best params:", random_search.best_params_)
print("RandomizedSearch best CV ROC-AUC:", random_search.best_score_)

Running RandomizedSearchCV...
RandomizedSearch best params: {'subsample': 0.6, 'reg_lambda': 1, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
RandomizedSearch best CV ROC-AUC: 0.850003416982591


In [ ]:
# -----------------------------
# 5. Optuna
# -----------------------------
print("Running Optuna...")


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.001, 0.2, log=True
        ),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 10.0, log=True
        ),
    }

    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[valid_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, proba))

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

start = time.perf_counter()
study.optimize(objective, n_trials=N_ITER)
optuna_time = time.perf_counter() - start

best_optuna_xgb = XGBClassifier(
    **study.best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_optuna_xgb.fit(X_train, y_train)

results.append(evaluate("Optuna", best_optuna_xgb, optuna_time))

print("Optuna best params:", study.best_params)
print("Optuna best CV ROC-AUC:", study.best_value)


Running Optuna...
Optuna best params: {'n_estimators': 424, 'max_depth': 2, 'learning_rate': 0.03330778827528297, 'subsample': 0.6971141625454798, 'colsample_bytree': 0.6245176990876247, 'min_child_weight': 6, 'reg_lambda': 6.513807769366131}
Optuna best CV ROC-AUC: 0.850820012815736


In [ ]:
# -----------------------------
# 6. Final comparison
# -----------------------------
results_df = pd.DataFrame(results)

metric_columns = [
    "method",
    "search_time_sec",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
    "false_negatives",
    "false_positives",
]

print("\n================ FINAL TEST-SET COMPARISON ================")
print(
    results_df[metric_columns]
    .sort_values("test_roc_auc", ascending=False)
    .to_string(index=False)
)


================ FINAL TEST-SET COMPARISON ================
            method  search_time_sec  test_accuracy  test_precision  test_recall  test_f1  test_roc_auc  false_negatives  false_positives
RandomizedSearchCV            51.11       0.809794        0.686620     0.521390 0.592705      0.849988              179               89
            Optuna            63.06       0.800568        0.657627     0.518717 0.579970      0.848772              180              101
      GridSearchCV            43.27       0.799148        0.658537     0.505348 0.571861      0.845593              185               98
     Basic XGBoost             2.39       0.774308        0.587500     0.502674 0.541787      0.818571              186              132
